# Delta System — MuSiQue Validation (Second Dataset)

**Purpose**: Prove the delta-system generalizes across dataset types, not just Wikipedia paragraphs.

**Data**: MuSiQue 2-hop QA dataset (Trivedi et al., 2022)
- A = first supporting paragraph (what is already known)
- B = first + second supporting paragraphs
- novel = second supporting paragraph (ground-truth novelty)

This is structurally different from Wikipedia consecutive paragraphs:
- Wikipedia: free-form encyclopedic text, paragraphs naturally follow each other
- MuSiQue: structured multi-hop reasoning, paragraphs are from different articles connected by a reasoning chain

**Pass criteria (held-out only)**: `DELTA_PPL > 2` AND `SPECIFICITY > 2`

**Compare to Wikipedia result**: DELTA_PPL +755, SPECIFICITY +608

In [ ]:
# Cell 1 — Install
!pip install transformers scikit-learn -q

In [ ]:
# Cell 2 — Clone / pull repo
import os, sys
from pathlib import Path

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

DELTA_DIR = REPO / 'delta_system'
sys.path.insert(0, str(DELTA_DIR))
print(f'delta_system: {DELTA_DIR}')
print(f'Files: {sorted([f.name for f in DELTA_DIR.glob("*.py")])}')

In [ ]:
# Cell 3 — Download MuSiQue data
# MuSiQue: Trivedi et al. 2022 — Multi-hop Questions via Single-hop Question Composition
# https://github.com/StonyBrookNLP/musique

DATA_DIR = Path('/kaggle/working/musique')
DATA_DIR.mkdir(exist_ok=True)
JSONL_PATH = DATA_DIR / 'musique_ans_v1.0_train.jsonl'

if not JSONL_PATH.exists():
    print('Downloading MuSiQue...')
    # Try primary URL
    ret = os.system(
        f'wget -q https://github.com/StonyBrookNLP/musique/releases/download/v1.0/musique_v1.0.zip '
        f'-O /kaggle/working/musique.zip'
    )
    if ret != 0:
        # Fallback URL variant
        os.system(
            f'wget -q https://github.com/StonyBrookNLP/musique/releases/download/v0.0.1/musique_v1.0.zip '
            f'-O /kaggle/working/musique.zip'
        )
    os.system(f'unzip -q /kaggle/working/musique.zip -d /kaggle/working/musique_raw')
    # Find the train JSONL
    found = list(Path('/kaggle/working/musique_raw').rglob('musique_ans_v1.0_train.jsonl'))
    if found:
        os.system(f'cp "{found[0]}" "{JSONL_PATH}"')
        print(f'Copied: {found[0]} -> {JSONL_PATH}')
    else:
        print('ERROR: could not find musique_ans_v1.0_train.jsonl in zip')
        print('Files found:', list(Path('/kaggle/working/musique_raw').rglob('*.jsonl')))
else:
    print(f'Already exists: {JSONL_PATH}')

print(f'Data file exists: {JSONL_PATH.exists()}')
if JSONL_PATH.exists():
    size_mb = JSONL_PATH.stat().st_size / 1e6
    print(f'File size: {size_mb:.1f} MB')

In [ ]:
# Cell 4 — Config
import torch

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN  = 5000
N_EVAL   = 500
STEPS    = 2000
BS       = 16
LR       = 1e-4
LAM_S    = 1.0
LAM_SPEC = 1.0
MARGIN   = 2.0
LOG_EVERY = 200
MAX_LEN  = 128

print(f'Device : {DEVICE}')
print(f'Config : {N_TRAIN} train / {N_EVAL} held-out | {STEPS} steps | bs={BS}')
print(f'Dataset: MuSiQue 2-hop (second dataset validation)')

In [ ]:
# Cell 5 — Load MuSiQue pairs
# A = hop-1 paragraph (known), B = hop-1 + hop-2, novel = hop-2 paragraph
# Only 2-hop questions — clean ground-truth novelty structure

import json

def load_musique_pairs(jsonl_path, n_train=5000, n_eval=500):
    total = n_train + n_eval
    pairs = []
    with open(jsonl_path, encoding='utf-8') as fh:
        for line in fh:
            ex = json.loads(line)
            decomp = ex.get('question_decomposition', [])
            if len(decomp) != 2:       # only 2-hop
                continue
            paras = {p['idx']: p['paragraph_text'].strip() for p in ex['paragraphs']}
            idx1  = decomp[0].get('paragraph_support_idx')
            idx2  = decomp[1].get('paragraph_support_idx')
            if idx1 is None or idx2 is None:
                continue
            A     = paras.get(idx1, '').strip()
            novel = paras.get(idx2, '').strip()
            if not A or not novel:
                continue
            pairs.append({'A': A, 'B': A + ' ' + novel, 'novel': novel})
            if len(pairs) >= total:
                break
    return pairs[:n_train], pairs[n_train:total]


train_pairs, eval_pairs = load_musique_pairs(JSONL_PATH, N_TRAIN, N_EVAL)
print(f'Train : {len(train_pairs)} pairs')
print(f'Eval  : {len(eval_pairs)} pairs (held-out)')
print(f'\nExample A     : {train_pairs[0]["A"][:120]}')
print(f'Example novel : {train_pairs[0]["novel"][:120]}')

# Sanity check: A and novel should NOT be identical
overlap = sum(1 for p in train_pairs[:50] if p['A'][:50] == p['novel'][:50])
print(f'\nA==novel overlap (should be 0): {overlap}/50')

In [ ]:
# Cell 6 — Load model + tokenizer (same architecture as Wikipedia experiment)
from model import DeltaSystem
from transformers import BertTokenizerFast

tok   = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = DeltaSystem().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params : {n_params/1e6:.1f}M')
print(f'BERT frozen      : {not next(model.bert.parameters()).requires_grad}')
print(f'Architecture     : identical to Wikipedia experiment')

In [ ]:
# Cell 7 — Train G on MuSiQue pairs
import math
from torch.utils.data import DataLoader, Dataset
from losses import recon_loss, sparsity_loss, specificity_loss

class PairDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self):         return len(self.pairs)
    def __getitem__(self, i):  return self.pairs[i]['A'], self.pairs[i]['B']

def make_collate(tok):
    def collate(batch):
        eA = tok([x[0] for x in batch], max_length=MAX_LEN, truncation=True,
                 padding='max_length', return_tensors='pt')
        eB = tok([x[1] for x in batch], max_length=MAX_LEN, truncation=True,
                 padding='max_length', return_tensors='pt')
        return eA['input_ids'], eA['attention_mask'], eB['input_ids'], eB['attention_mask']
    return collate

dl  = DataLoader(PairDS(train_pairs), batch_size=BS, shuffle=True,
                 collate_fn=make_collate(tok), num_workers=2, pin_memory=True)
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

model.train()
step = 0
while step < STEPS:
    for batch in dl:
        if step >= STEPS: break
        A_ids, A_mask, B_ids, B_mask = [t.to(DEVICE) for t in batch]
        b = A_ids.size(0)

        logits, delta, delta_0, H_A, alpha = model(A_ids, A_mask, B_ids, B_mask)
        L_r    = recon_loss(logits, B_ids, B_mask)
        L_s    = sparsity_loss(delta, B_mask)
        L_spec = torch.tensor(0.0, device=DEVICE)
        if b > 1:
            idx_shift    = list(range(1, b)) + [0]
            logits_wrong = model.reconstruct(
                H_A, A_mask, delta[idx_shift], delta_0[idx_shift], B_ids, B_mask)
            L_spec = specificity_loss(logits, logits_wrong, B_ids, B_mask, margin=MARGIN)

        loss = L_r + LAM_S * L_s + LAM_SPEC * L_spec
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        step += 1

        if step % LOG_EVERY == 0 or step == 1:
            ppl = math.exp(min(L_r.item(), 20))
            print(f'  step {step:4d}/{STEPS} | ppl={ppl:.1f} | L_spec={L_spec.item():.4f}')

print('Training complete.')

In [ ]:
# Cell 8 — Save checkpoint
ckpt_dir = Path('/kaggle/working/checkpoints')
ckpt_dir.mkdir(exist_ok=True)
trainable = {k: v for k, v in model.state_dict().items() if not k.startswith('bert.')}
torch.save(trainable, ckpt_dir / 'musique_model.pt')
print(f'Saved: {ckpt_dir / "musique_model.pt"}')

In [ ]:
# Cell 9 — HELD-OUT EVALUATION (key result)
from eval import evaluate

print('=' * 62)
print(f'HELD-OUT EVAL — {len(eval_pairs)} MuSiQue pairs never seen during training')
print('=' * 62)
held_out_results = evaluate(model, eval_pairs, tok)

In [ ]:
# Cell 10 — In-sample check + cross-dataset comparison
print('=' * 62)
print('IN-SAMPLE CHECK — 200 training pairs')
print('=' * 62)
train_results = evaluate(model, train_pairs[:200], tok)

print('\n' + '=' * 62)
print('CROSS-DATASET COMPARISON')
print('=' * 62)
print('                        DELTA_PPL    SPECIFICITY')
print('Wikipedia held-out  :    +755          +608      (8000 train, 1000 held-out)')
print(f'MuSiQue   held-out  :  {held_out_results["delta_ppl"]:+.0f}        '
      f'{held_out_results["specificity"]:+.0f}      (5000 train, 500 held-out)')
print()
print('Key: Both datasets should PASS with positive values.')
print('     If both pass → architecture generalizes across dataset types.')
print()

if held_out_results['pass']:
    print('MuSiQue HELD-OUT: PASS')
    print('The delta-system generalizes to multi-hop reasoning structure (MuSiQue)')
    print('as well as encyclopedic text (Wikipedia). Two-dataset validation complete.')
else:
    print('MuSiQue HELD-OUT: FAIL')
    if held_out_results['delta_ppl'] <= 2:
        print('  -> DELTA_PPL failed: try increasing STEPS or N_TRAIN')
    if held_out_results['specificity'] <= 2:
        print('  -> SPECIFICITY failed: try increasing LAM_SPEC or MARGIN')
print('=' * 62)